# TEP analysis: re-run every analysis of the paper

Runs the analyses of the manuscript in order, on your data, in Google Colab.

1. Put the folder `code/` and your input files in one Drive folder (default `MyDrive/TEP_analysis/`):
   `dataset_NSCLC`, `GSE89843_filtered_metadata.csv`, `DEFINITIVE_biomarker_panel.csv`,
   `GSE68086_TEP_data_matrix.txt.gz` and `dgeGenesEnsembl75.RData`. The last one is in `bin/` of
   https://github.com/MyronBest/thromboSeq_source_code.
2. Run the cells from top to bottom. Each long step saves its results, so after a disconnect you can
   simply run the cell again.
3. At the end, copy `out/results_macros.tex`, `out/figures/` and `out/tables/` into `manuscript/results/`
   and recompile the paper.

In [ ]:
!pip -q install xgboost catboost shap statsmodels inmoose gseapy pyreadr pyarrow adjustText
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/TEP_analysis'      # <- edit if needed
os.chdir(BASE)
os.makedirs('inputs/gse68086', exist_ok=True)

## 1. Inputs

In [ ]:
!python code/prepare_inputs.py dataset_NSCLC GSE89843_filtered_metadata.csv inputs/
!cp DEFINITIVE_biomarker_panel.csv inputs/
!python code/prepare_gse68086.py GSE68086_TEP_data_matrix.txt.gz dgeGenesEnsembl75.RData inputs/gse68086/

Optional, for the pathway analysis: put MSigDB v7.5.1 `.gmt` files in `inputs/genesets/`. Their paths are
listed in `code/config.json` under `gene_sets`.

## 2. Reproduction of the original selection (about 20 min)

In [ ]:
!python code/extra_analyses.py code/config.json repro

## 3. Discovery-cohort analyses (about 1 hour)

In [ ]:
!python code/run_steps.py code/config.json r0 r1t r1g nested nestedfig r8 r4 r1v r1n r2e

In [ ]:
!python code/run_steps.py code/config.json r5 r6 shap

In [ ]:
# pathway analysis: one gene-set library per process (keeps memory below about 3.5 GB)
import json
gene_sets = json.load(open('code/config.json'))['gene_sets']
!python code/r7_gsea.py code/config.json rank
for g in gene_sets:
    !python code/r7_gsea.py code/config.json lib {g} 1000 500
!python code/r7_gsea.py code/config.json ora
!python code/r7_gsea.py code/config.json combine
# sensitivity: ranking adjusted for the leukocyte, skin and erythroid marker scores
!python code/r7_gsea.py code/config.json rank_adj
for g in gene_sets:
    !python code/r7_gsea.py code/config.json lib {g} 1000 500 adj
!python code/r7_gsea.py code/config.json combine_adj

In [ ]:
for s in ['libsize', 'r0extra', 'nestedextra', 'subsvumc', 'r5extra', 'weights']:
    !python code/extra_analyses.py code/config.json {s}

In [ ]:
# ablation, feature-count curve, RFECV ceiling and stability selection (about 35 min)
for s in ['ablation', 'featcount', 'rfecv']:
    !python code/dev_reruns.py code/config.json {s}

## 3b. Split-half replication of genes and pathways (about 50 min)

In [ ]:
!python code/pathway_stability.py code/config.json run 25 500
!python code/pathway_stability.py code/config.json full 1000
!python code/pathway_stability.py code/config.json summarise

## 3c. Fold-recomputable nested CV: feature-importance screen (and, with STRING, the PPI term) re-derived in every training fold (about 1 h)

Colab has internet access, so the STRING step runs here and the `ppi` step reports the five-term and the six-term estimates.

In [ ]:
!python code/fold_recompute.py code/config.json fi_full
!python code/fold_recompute.py code/config.json nested
!python code/fold_recompute.py code/config.json mrmr_union
!cp out/tables/FR_mrmr_union_genes.txt FR_string_query_genes.txt
!python code/get_string_edges.py
!python code/fold_recompute.py code/config.json ppi string_edges.tsv

## 3d. Sensitivity of the re-implemented comparator (about 5 min)

In [ ]:
!python code/sheng_sensitivity.py code/config.json run
!python code/sheng_sensitivity.py code/config.json summarise

## 4. External cohort and harmonisation (about 40 min)

In [ ]:
!python code/external_analysis.py code/config_ext.json
!python code/run_steps.py code/config_ext.json r3
!python code/r5_external_table.py code/config_ext.json
!python code/harm_sim.py code/config.json 30 50
!python code/ext_reliability.py code/config_ext.json   # locked model: calibration and gene availability

## 5. Random and matched null panels (longest step: about 2 hours on 2 cores; restartable)

In [ ]:
!python code/run_steps.py code/config_r9.json r9

Optional: compare with the second, independently written notebook (`TEP_Reviewer_Revision_Run.ipynb`). Unzip its PAPER results folder next to `out/` and give its name below.

In [ ]:
!python code/crosscheck_independent.py out PAPER_2865dafe4618

## 6. Figures, merge, done

In [ ]:
!python code/run_steps.py code/config.json merge
!python code/make_forest.py out
!python code/make_fig10.py out 0.8481
!python code/redraw_r1_effects.py out
!python code/redraw_r3sim.py out
!python code/redraw_r9.py out
print(open('out/results_macros.tex').read()[:2000])
!python code/make_fig01_pmtrd.py